In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
     Implement a quantized matrix multiplication program for 8-bit signed integer matrices. Given two input matrices <code>A</code> of dimensions $M \times K$ and <code>B</code> of dimensions $K \times N$, quantization scales <code>scale_A</code>, <code>scale_B</code>, output scale <code>scale_C</code>, zero-points <code>zero_point_A</code>, <code>zero_point_B</code>, <code>zero_point_C</code>, compute:
     $$
     C_{\text{quant}}(i, j) = \mathrm{clamp}\left(
         \mathrm{round}\left(
             \frac{
                 \sum_{k=0}^{K-1} (A_{ik} - z_A)(B_{kj} - z_B) \cdot s_A s_B
             }{s_C}
         \right) + z_C,\ -128,\ 127
     \right)
     $$
     where <code>s_A = scale_A</code>, <code>z_A = zero_point_A</code>, etc.
     </p>

     <h2>Implementation Requirements</h2>
     <ul>
     <li>External libraries are not permitted</li>
     <li>The <code>solve</code> function signature must remain unchanged</li>
     <li>The final result must be stored in the output matrix <code>C</code> as <code>int8</code></li>
     <li>After accumulation in int32 and scaling in float32, values must be rounded to the nearest integer, shifted by <code>zero_point_C</code>, and clamped to the <code>[-128, 127]</code> range</li>
     </ul>

     <h2>Example 1:</h2>
     <pre>
     Input:
     A = [[1, 2],
          [3, 4]]
     B = [[5, 6],
          [7, 8]]
     M = 2, N = 2, K = 2
     scale_A = 0.1, scale_B = 0.2, scale_C = 0.05
     zero_point_A = 0, zero_point_B = 0, zero_point_C = 0

     Output:
     C = [[19, 22],
          [43, 50]]
     </pre>

     <h2>Example 2:</h2>
     <pre>
     Input:
     A = [[1, 2]]
     B = [[3],
          [4]]
     M = 1, N = 1, K = 2
     scale_A = 1.0, scale_B = 1.0, scale_C = 1.0
     zero_point_A = 1, zero_point_B = 3, zero_point_C = 5

     Output:
     C = [[6]]
     </pre>

     <h2>Constraints</h2>
     <ul>
     <li>1 ≤ <code>M</code>, <code>N</code>, <code>K</code> ≤ 4096</li>
     <li><code>scale_A</code>, <code>scale_B</code>, <code>scale_C</code> are positive floats</li>
     <li><code>-128</code> ≤ <code>zero_point_A</code>, <code>zero_point_B</code>, <code>zero_point_C</code> ≤ <code>127</code></li>

  <li>Performance is measured with <code>K</code> = 2,048, <code>M</code> = 8,192, <code>N</code> = 4,096</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, C are device pointers
extern "C" void solve(const int8_t* A, const int8_t* B, int8_t* C, int M, int N, int K,
                      float scale_A, float scale_B, float scale_C, int zero_point_A,
                      int zero_point_B, int zero_point_C) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    scale_A: cute.Float32,
    scale_B: cute.Float32,
    scale_C: cute.Float32,
    zero_point_A: cute.Int32,
    zero_point_B: cute.Int32,
    zero_point_C: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(
    A: jax.Array,
    B: jax.Array,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Int8, MutExternalOrigin],
    B: UnsafePointer[Int8, MutExternalOrigin],
    C: UnsafePointer[Int8, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    scale_A: Float32,
    scale_B: Float32,
    scale_C: Float32,
    zero_point_A: Int32,
    zero_point_B: Int32,
    zero_point_C: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, C are tensors on the GPU
def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# a, b, c are tensors on the GPU
def solve(
    a: torch.Tensor,
    b: torch.Tensor,
    c: torch.Tensor,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="INT8 Quantized MatMul", atol=0, rtol=0, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        A: torch.Tensor,
        B: torch.Tensor,
        C: torch.Tensor,
        M: int,
        N: int,
        K: int,
        scale_A: float,
        scale_B: float,
        scale_C: float,
        zero_point_A: int,
        zero_point_B: int,
        zero_point_C: int,
    ):
        A = A.view(M, K).to(torch.int32)
        B = B.view(K, N).to(torch.int32)
        A_f = (A - zero_point_A).to(torch.float32)
        B_f = (B - zero_point_B).to(torch.float32)
        C_f = torch.matmul(A_f, B_f).round().int()  # closest thing to integer accumulation we have
        C_f = C_f * scale_A * scale_B / scale_C
        C_q = torch.round(C_f).to(torch.int32) + zero_point_C
        C_q = torch.clamp(C_q, -128, 127).to(torch.int8)
        C.view(M, N).copy_(C_q)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_int8), "in"),
            "B": (ctypes.POINTER(ctypes.c_int8), "in"),
            "C": (ctypes.POINTER(ctypes.c_int8), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "K": (ctypes.c_int, "in"),
            "scale_A": (ctypes.c_float, "in"),
            "scale_B": (ctypes.c_float, "in"),
            "scale_C": (ctypes.c_float, "in"),
            "zero_point_A": (ctypes.c_int, "in"),
            "zero_point_B": (ctypes.c_int, "in"),
            "zero_point_C": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.int8
        device = "cuda"
        A = torch.tensor([[1, 2], [3, 4]], dtype=dtype, device=device).flatten()
        B = torch.tensor([[5, 6], [7, 8]], dtype=dtype, device=device).flatten()
        C = torch.tensor([[0, 0], [0, 0]], dtype=dtype, device=device).flatten()

        return {
            "A": A,
            "B": B,
            "C": C,
            "M": 2,
            "N": 2,
            "K": 2,
            "scale_A": 0.1,
            "scale_B": 0.2,
            "scale_C": 0.05,
            "zero_point_A": 0,
            "zero_point_B": 0,
            "zero_point_C": 0,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.int8
        device = "cuda"
        tests = []

        # 1. 4x4x4_zero_zp
        A1 = torch.randint(-128, 128, (4, 4), dtype=dtype, device=device)
        B1 = torch.randint(-128, 128, (4, 4), dtype=dtype, device=device)
        C1 = torch.randint(-128, 128, (4, 4), dtype=dtype, device=device)
        tests.append(
            {
                "A": A1,
                "B": B1,
                "C": C1,
                "M": 4,
                "N": 4,
                "K": 4,
                "scale_A": 0.1,
                "scale_B": 0.2,
                "scale_C": 0.05,
                "zero_point_A": 0,
                "zero_point_B": 0,
                "zero_point_C": 0,
            }
        )

        # 2. 2x3x5_nonzero_zp
        A2 = torch.randint(-128, 128, (2, 5), dtype=dtype, device=device)
        B2 = torch.randint(-128, 128, (5, 3), dtype=dtype, device=device)
        C2 = torch.empty((2, 3), dtype=dtype, device=device)
        tests.append(
            {
                "A": A2,
                "B": B2,
                "C": C2,
                "M": 2,
                "N": 3,
                "K": 5,
                "scale_A": 0.5,
                "scale_B": 0.25,
                "scale_C": 0.125,
                "zero_point_A": 1,
                "zero_point_B": -2,
                "zero_point_C": 3,
            }
        )

        # 3. 1x1x3
        A3 = torch.randint(-128, 128, (1, 3), dtype=dtype, device=device)
        B3 = torch.randint(-128, 128, (3, 1), dtype=dtype, device=device)
        C3 = torch.empty((1, 1), dtype=dtype, device=device)
        tests.append(
            {
                "A": A3,
                "B": B3,
                "C": C3,
                "M": 1,
                "N": 1,
                "K": 3,
                "scale_A": 1.0,
                "scale_B": 1.0,
                "scale_C": 1.0,
                "zero_point_A": 1,
                "zero_point_B": 3,
                "zero_point_C": 5,
            }
        )

        # 4. 3x5x2
        A4 = torch.randint(-50, 51, (3, 2), dtype=dtype, device=device)
        B4 = torch.randint(-50, 51, (2, 5), dtype=dtype, device=device)
        C4 = torch.zeros((3, 5), dtype=dtype, device=device)
        tests.append(
            {
                "A": A4,
                "B": B4,
                "C": C4,
                "M": 3,
                "N": 5,
                "K": 2,
                "scale_A": 0.05,
                "scale_B": 0.1,
                "scale_C": 0.01,
                "zero_point_A": 0,
                "zero_point_B": 0,
                "zero_point_C": 0,
            }
        )

        # 5. 32x32x16
        A5 = torch.randint(-128, 128, (32, 16), dtype=dtype, device=device)
        B5 = torch.randint(-128, 128, (16, 32), dtype=dtype, device=device)
        C5 = torch.empty((32, 32), dtype=dtype, device=device)
        tests.append(
            {
                "A": A5,
                "B": B5,
                "C": C5,
                "M": 32,
                "N": 32,
                "K": 16,
                "scale_A": 0.2,
                "scale_B": 0.3,
                "scale_C": 0.1,
                "zero_point_A": 0,
                "zero_point_B": 0,
                "zero_point_C": 0,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.int8
        device = "cuda"
        shape_A = (8192, 2048)
        shape_B = (2048, 4096)
        shape_C = (8192, 4096)
        A = torch.randint(-128, 128, (shape_A[0] * shape_A[1],), dtype=dtype, device=device)
        B = torch.randint(-128, 128, (shape_B[0] * shape_B[1],), dtype=dtype, device=device)
        C = torch.empty(shape_C[0] * shape_C[1], dtype=dtype, device=device)
        M = 8192
        N = 4096
        K = 2048
        scale_A = 0.1
        scale_B = 0.1
        scale_C = 0.01
        zero_point_A = 0
        zero_point_B = 0
        zero_point_C = 0
        return {
            "A": A,
            "B": B,
            "C": C,
            "M": M,
            "N": N,
            "K": K,
            "scale_A": scale_A,
            "scale_B": scale_B,
            "scale_C": scale_C,
            "zero_point_A": zero_point_A,
            "zero_point_B": zero_point_B,
            "zero_point_C": zero_point_C,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
